# 09-01B. Audited Label Revalidation
## 09-01A 보정이 Model A / Model B에 미치는 영향 검증

---

09-01A 결과에서 보수적 자동 교정은 **20건**이었습니다.

이번 Notebook의 목적은:

```text
원본 label
vs
09-01A audited label
```

을 동일한 walk-forward protocol로 비교하는 것입니다.

---

# 왜 09-02로 바로 가지 않는가?

09-01A에서 확인한 것은:

- `matched_next=False` 중 일부는 명백한 season-to-season 매칭 오류
- 하지만 `STABLE_BIG5_REVIEW` 전체를 오류라고 볼 수는 없음
- 강등 팀 잔류, 후반 이적, 부상, 은퇴, raw-source 누락 등 다양한 이유가 섞임

따라서:

> **확실하게 고친 20건만 반영했을 때 모델 결론이 달라지는지**

먼저 봅니다.

---

# 이번 Notebook에서 재검증하는 두 모델

## Model A — Big5 Presence Classifier

```text
P(next-season Big5 record exists)
```

09-01 champion이었던 fixed CatBoost classifier를 재검증합니다.

비교:

```text
A-Original
target = matched_next_original

A-Audited
target = matched_next_audited
```

## Model B — Conditional Goals Regressor

08-03에서 external-context 대표 모델이었던:

```text
Fixed CatBoost S3
```

를 같은 방식으로 재검증합니다.

비교:

```text
B-Original
population = matched_next_original=True
target     = next_goals_original

B-Audited
population = matched_next_audited=True
target     = next_goals_audited
```

---

# Final Test

```text
2024-25 → 2025-26
```

는 계속 LOCKED 상태입니다.

In [13]:
from pathlib import Path

filename = "09_01A_snapshot_conservative_labels.csv"

print("=== Google Drive ===")
for p in Path("/content/drive/MyDrive").rglob(filename):
    print(p)

print("\n=== Colab runtime ===")
for p in Path("/content").rglob(filename):
    print(p)

=== Google Drive ===
/content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_snapshot_conservative_labels.csv

=== Colab runtime ===
/content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_snapshot_conservative_labels.csv


In [14]:
from pathlib import Path

filename = "09_01A_snapshot_conservative_labels.csv"

matches = list(
    Path("/content/drive/MyDrive").rglob(filename)
)

print("찾은 개수:", len(matches))

for p in matches:
    print(p)

찾은 개수: 1
/content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_snapshot_conservative_labels.csv


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 0. VS Code Colab / Google Drive

이 Notebook은 VS Code + Google Colab extension 기준입니다.

Notebook 내부에서 `drive.mount()`를 호출하지 않습니다.

먼저 VS Code에서:

```text
Ctrl + Shift + P
→ Colab: Mount Google Drive to Server...
```

를 실행하세요.

In [19]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive가 마운트되어 있지 않습니다.\n"
        "VS Code에서 Ctrl + Shift + P → "
        "'Colab: Mount Google Drive to Server...' 실행 후 "
        "다시 시작하세요."
    )

print(
    "Google Drive mounted:",
    DRIVE_ROOT.exists(),
)

Google Drive mounted: True


In [20]:
import json
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
    roc_auc_score,
)

warnings.filterwarnings(
    "ignore"
)

SEED = 42

random.seed(
    SEED
)

np.random.seed(
    SEED
)

pd.set_option(
    "display.max_columns",
    100,
)

## 1. CatBoost / GPU

In [21]:
try:
    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )

except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "catboost",
        ]
    )

    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )


try:
    import torch

    USE_GPU = bool(
        torch.cuda.is_available()
    )

except ImportError:
    USE_GPU = False


CATBOOST_DEVICE_PARAMS = (
    {
        "task_type": "GPU",
        "devices": "0",
    }
    if USE_GPU
    else {
        "task_type": "CPU",
    }
)

print(
    "CatBoost:",
    catboost.__version__,
)

print(
    "GPU:",
    USE_GPU,
)

print(
    CATBOOST_DEVICE_PARAMS
)

CatBoost: 1.2.10
GPU: True
{'task_type': 'GPU', 'devices': '0'}


# Part A. Load audited snapshot

In [22]:
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/"
    "next_season_goal_prediction (1)/"
    "artifacts"
)

SNAPSHOT_PATH = (
    ARTIFACT_DIR
    / "09_01A_snapshot_conservative_labels.csv"
)

if not SNAPSHOT_PATH.exists():
    raise FileNotFoundError(
        f"파일이 없습니다: {SNAPSHOT_PATH}"
    )

df = pd.read_csv(
    SNAPSHOT_PATH,
    low_memory=False,
)

print(
    "Shape:",
    df.shape,
)

print(
    "Audited changes:",
    int(
        df[
            "label_audit_changed"
        ].sum()
    ),
)

display(
    df[
        df[
            "label_audit_changed"
        ].eq(1)
    ][
        [
            "player",
            "season",
            "target_season",
            "team",
            "matched_next_original",
            "matched_next_audited",
            "next_goals_original",
            "next_goals_audited",
            "label_audit_rule",
        ]
    ]
)

Shape: (23353, 124)
Audited changes: 20


,player,season,target_season,team,matched_next_original,matched_next_audited,next_goals_original,next_goals_audited,label_audit_rule
20600,Amine Adli,2021-2022,2022-2023,Leverkusen,False,True,0.0,5.0,exact_name_age
20601,Amine Gouiri,2021-2022,2022-2023,Nice,False,True,0.0,15.0,exact_name_age
20735,Demarai Gray,2021-2022,2022-2023,Everton,False,True,0.0,4.0,exact_name_age
20885,Hugo Guillamón,2021-2022,2022-2023,Valencia,False,True,0.0,1.0,exact_name_age
20892,Ibrahima Sissoko,2021-2022,2022-2023,Strasbourg,False,True,0.0,0.0,exact_name_age
20975,Jonathan Bamba,2021-2022,2022-2023,Lille,False,True,0.0,6.0,exact_name_age
21031,Khéphren Thuram-Ulie,2021-2022,2022-2023,Nice,False,True,0.0,2.0,same_team_alias_age
21053,Lee Kangin,2021-2022,2022-2023,Mallorca,False,True,0.0,6.0,same_team_alias_age
21126,Martinelli,2021-2022,2022-2023,Arsenal,False,True,0.0,15.0,same_team_alias_age
21367,Sepe Elye Wahi,2021-2022,2022-2023,Montpellier,False,True,0.0,19.0,same_team_alias_age


## 2. Final Test Lock

In [23]:
LOCKED_TEST_INPUT_SEASON = (
    "2024-2025"
)

LOCKED_TEST_TARGET_SEASON = (
    "2025-2026"
)

assert (
    LOCKED_TEST_INPUT_SEASON
    not in set(
        df[
            "season"
        ].astype(str)
    )
)

print(
    "✅ Final Test locked:",
    LOCKED_TEST_INPUT_SEASON,
    "→",
    LOCKED_TEST_TARGET_SEASON,
)

✅ Final Test locked: 2024-2025 → 2025-2026


# Part B. Historical feature 재생성

08 단계와 동일하게
현재 row 이전 3시즌만 rolling history로 사용합니다.

In [24]:
df[
    "season_start"
] = (
    df[
        "season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

df[
    "target_year"
] = (
    df[
        "target_season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

df = (
    df
    .sort_values(
        [
            "player",
            "season_start",
        ]
    )
    .reset_index(
        drop=True
    )
)

player_group = (
    df.groupby(
        "player",
        sort=False,
    )
)

df[
    "goals_3yr_mean"
] = player_group[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_per90_3yr_mean"
] = player_group[
    "goals_per90"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_3yr_max"
] = player_group[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .max()
)

HISTORICAL_COLS = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max",
]

df[
    HISTORICAL_COLS
] = (
    df[
        HISTORICAL_COLS
    ]
    .fillna(0.0)
)

# Part C. Feature Sets

In [25]:
BASE_NUMERIC = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90",
]

TEAM_BASE_FEATURES = [
    "old_team_rank_pct",
    "old_team_points_per_game",
    "old_team_goal_diff_per_game",
]

MARKET_VALUE_FEATURES = [
    "market_value_known",
    "log_market_value",
    "market_value_percentile",
    "market_value_vs_position_median",
]

MARKET_MOMENTUM_FEATURES = [
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
]

TRANSFER_STATE_FEATURES = [
    "changed_team_preseason",
    "same_league_transfer",
    "league_changed",
    "country_changed",
    "is_loan_preseason",
    "days_since_transfer",
]

NEW_TEAM_FEATURES = [
    "new_team_prev_rank_pct",
    "new_team_prev_points_per_game",
    "new_team_prev_goal_diff_per_game",
    "team_rank_change",
    "team_points_change",
    "team_goal_diff_change",
    "new_team_strength_missing",
]

CATEGORICAL = [
    "league",
    "position_group",
]

S3_NUMERIC = (
    BASE_NUMERIC
    + HISTORICAL_COLS
    + TEAM_BASE_FEATURES
    + MARKET_VALUE_FEATURES
    + MARKET_MOMENTUM_FEATURES
    + TRANSFER_STATE_FEATURES
    + NEW_TEAM_FEATURES
)

MODEL_A_SPECIFIC = [
    "transfer_event_preseason",
    "destination_in_big5",
]

MODEL_A_NUMERIC = (
    S3_NUMERIC
    + MODEL_A_SPECIFIC
)

MODEL_A_FEATURES = (
    MODEL_A_NUMERIC
    + CATEGORICAL
)

MODEL_B_FEATURES = (
    S3_NUMERIC
    + CATEGORICAL
)

print(
    "Model A raw features:",
    len(
        MODEL_A_FEATURES
    ),
)

print(
    "Model B raw features:",
    len(
        MODEL_B_FEATURES
    ),
)

Model A raw features: 41
Model B raw features: 39


## 3. Leakage guard

In [26]:
FORBIDDEN = {
    "matched_next",
    "matched_next_original",
    "matched_next_audited",
    "next_goals",
    "next_goals_original",
    "next_goals_audited",
    "next_10plus",
    "next_10plus_original",
    "next_10plus_audited",
    "label_audit_changed",
    "label_audit_rule",
}

assert not (
    FORBIDDEN
    & set(
        MODEL_A_FEATURES
    )
)

assert not (
    FORBIDDEN
    & set(
        MODEL_B_FEATURES
    )
)

print(
    "✅ Label columns are not model features."
)

✅ Label columns are not model features.


# Part D. Audit 영향 먼저 확인

In [27]:
dev = (
    df[
        df[
            "target_year"
        ].ge(2017)
    ]
    .copy()
)

changed_dev = (
    dev[
        dev[
            "label_audit_changed"
        ].eq(1)
    ]
)

impact_summary = pd.DataFrame({
    "metric": [
        "development_rows",
        "corrections",
        "presence_before",
        "presence_after",
        "presence_rate_before",
        "presence_rate_after",
        "recovered_goal_sum",
        "recovered_10plus_n",
    ],
    "value": [
        len(
            dev
        ),
        len(
            changed_dev
        ),
        dev[
            "matched_next_original"
        ].sum(),
        dev[
            "matched_next_audited"
        ].sum(),
        dev[
            "matched_next_original"
        ].mean(),
        dev[
            "matched_next_audited"
        ].mean(),
        (
            changed_dev[
                "next_goals_audited"
            ]
            - changed_dev[
                "next_goals_original"
            ]
        ).sum(),
        changed_dev[
            "next_goals_audited"
        ].ge(10).sum(),
    ],
})

impact_summary

,metric,value
0,development_rows,7572.000000
1,corrections,20.000000
2,presence_before,6371.000000
3,presence_after,6391.000000
4,presence_rate_before,0.841389
5,presence_rate_after,0.844031
6,recovered_goal_sum,95.000000
7,recovered_10plus_n,3.000000


## 4. `STABLE_BIG5_REVIEW`가 전부 오류는 아님

09-01A에서 `STABLE_BIG5_REVIEW`가 많았지만,
여기에는 **강등 팀 잔류 선수**가 다수 포함될 가능성이 큽니다.

이를 보기 위해 old-team rank를 diagnostic으로만 확인합니다.

`old_team_rank_pct`는:

```text
0에 가까울수록 하위권
1에 가까울수록 상위권
```

입니다.

이 셀은 feature 추가가 아니라 **audit 설명용**입니다.

In [28]:
AUDIT_PATH = (
    ARTIFACT_DIR
    / "09_01A_matched_next_label_audit.csv"
)

if AUDIT_PATH.exists():
    audit = pd.read_csv(
        AUDIT_PATH,
        low_memory=False,
    )

    stable = (
        audit[
            audit[
                "audit_status"
            ].eq(
                "STABLE_BIG5_REVIEW"
            )
        ]
        .copy()
    )

    stable_rank_diag = pd.DataFrame({
        "metric": [
            "stable_review_n",
            "rank_pct_le_0.05",
            "rank_pct_le_0.10",
            "rank_pct_le_0.15",
            "rank_pct_gt_0.15",
        ],
        "n": [
            len(
                stable
            ),
            stable[
                "old_team_rank_pct"
            ].le(
                0.05
            ).sum(),
            stable[
                "old_team_rank_pct"
            ].le(
                0.10
            ).sum(),
            stable[
                "old_team_rank_pct"
            ].le(
                0.15
            ).sum(),
            stable[
                "old_team_rank_pct"
            ].gt(
                0.15
            ).sum(),
        ],
    })

    stable_rank_diag[
        "rate"
    ] = (
        stable_rank_diag[
            "n"
        ]
        / len(
            stable
        )
    )

    display(
        stable_rank_diag
    )

else:
    stable_rank_diag = (
        pd.DataFrame()
    )

    print(
        "09_01A_matched_next_label_audit.csv 없음"
    )

,metric,n,rate
0,stable_review_n,836,1.000000
1,rank_pct_le_0.05,266,0.318182
2,rank_pct_le_0.10,528,0.631579
3,rank_pct_le_0.15,706,0.844498
4,rank_pct_gt_0.15,130,0.155502


# Part E. Shared temporal protocol

In [29]:
OUTER_VAL_SEASONS = [
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
]


def split_outer(
    data,
    val_season,
):
    val_start = int(
        val_season[:4]
    )

    train_df = (
        data[
            data[
                "season_start"
            ]
            < val_start
        ]
        .copy()
    )

    val_df = (
        data[
            data[
                "season_start"
            ]
            == val_start
        ]
        .copy()
    )

    return (
        train_df,
        val_df,
    )


def split_calibration(
    outer_train_df,
):
    cal_val_start = (
        outer_train_df[
            "season_start"
        ].max()
    )

    cal_train = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            < cal_val_start
        ]
        .copy()
    )

    cal_val = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            == cal_val_start
        ]
        .copy()
    )

    if (
        cal_train.empty
        or cal_val.empty
    ):
        raise ValueError(
            "Calibration split failure"
        )

    return (
        cal_train,
        cal_val,
    )

# Part F. CatBoost data preparation

In [30]:
def prepare_X(
    data,
    numeric_features,
):
    X = (
        data[
            numeric_features
            + CATEGORICAL
        ]
        .copy()
    )

    for col in (
        CATEGORICAL
    ):
        X[col] = (
            X[col]
            .fillna(
                "__MISSING__"
            )
            .astype(str)
        )

    for col in (
        numeric_features
    ):
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce",
        )

    cat_indices = [
        X.columns.get_loc(
            col
        )
        for col in (
            CATEGORICAL
        )
    ]

    return (
        X,
        cat_indices,
    )

# Part G. Model A revalidation

## 5. Classifier protocol

09-01과 동일합니다.

```text
depth = 6
learning_rate = 0.03
loss = Logloss
max iterations = 2000
early stopping = 60
```

Threshold:

```text
Outer Train의 마지막 시즌
→ Macro F1 최적 threshold
```

In [31]:
A_MAX_ITER = 2000
A_PATIENCE = 60

A_FIXED_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
}


def class_probability_metrics(
    y_true,
    prob,
):
    p = np.clip(
        np.asarray(
            prob,
            dtype=float,
        ),
        1e-7,
        1 - 1e-7,
    )

    y = np.asarray(
        y_true,
        dtype=int,
    )

    return {
        "log_loss": log_loss(
            y,
            p,
            labels=[
                0,
                1,
            ],
        ),
        "brier": (
            brier_score_loss(
                y,
                p,
            )
        ),
        "roc_auc": (
            roc_auc_score(
                y,
                p,
            )
        ),
        "pr_auc_presence": (
            average_precision_score(
                y,
                p,
            )
        ),
        "pr_auc_exit": (
            average_precision_score(
                1 - y,
                1 - p,
            )
        ),
    }


def class_threshold_metrics(
    y_true,
    prob,
    threshold,
):
    y = np.asarray(
        y_true,
        dtype=int,
    )

    pred = (
        np.asarray(
            prob
        )
        >= threshold
    ).astype(int)

    (
        precision,
        recall,
        f1,
        support,
    ) = (
        precision_recall_fscore_support(
            y,
            pred,
            labels=[
                0,
                1,
            ],
            zero_division=0,
        )
    )

    return {
        "accuracy": (
            accuracy_score(
                y,
                pred,
            )
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y,
                pred,
            )
        ),
        "macro_f1": (
            f1_score(
                y,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "exit_precision": (
            precision[0]
        ),
        "exit_recall": (
            recall[0]
        ),
        "exit_f1": (
            f1[0]
        ),
        "presence_precision": (
            precision[1]
        ),
        "presence_recall": (
            recall[1]
        ),
        "presence_f1": (
            f1[1]
        ),
    }


THRESHOLD_GRID = np.round(
    np.arange(
        0.20,
        0.951,
        0.01,
    ),
    2,
)


def choose_threshold(
    y_true,
    prob,
):
    rows = []

    for th in (
        THRESHOLD_GRID
    ):
        m = (
            class_threshold_metrics(
                y_true,
                prob,
                th,
            )
        )

        m[
            "threshold"
        ] = th

        rows.append(
            m
        )

    table = pd.DataFrame(
        rows
    )

    best = (
        table
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "presence_recall",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    return (
        float(
            best[
                "threshold"
            ]
        ),
        table,
    )

In [32]:
def fit_model_a(
    outer_train_df,
    outer_val_df,
    target_col,
    seed,
):
    (
        cal_train,
        cal_val,
    ) = split_calibration(
        outer_train_df
    )

    (
        X_cal_train,
        cat_indices,
    ) = prepare_X(
        cal_train,
        MODEL_A_NUMERIC,
    )

    (
        X_cal_val,
        _,
    ) = prepare_X(
        cal_val,
        MODEL_A_NUMERIC,
    )

    selector = (
        CatBoostClassifier(
            iterations=A_MAX_ITER,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **A_FIXED_PARAMS,
        )
    )

    selector.fit(
        X_cal_train,
        cal_train[
            target_col
        ].astype(int),
        cat_features=(
            cat_indices
        ),
        eval_set=(
            X_cal_val,
            cal_val[
                target_col
            ].astype(int),
        ),
        early_stopping_rounds=(
            A_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    selected_iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    cal_prob = (
        selector.predict_proba(
            X_cal_val
        )[:, 1]
    )

    (
        threshold,
        threshold_table,
    ) = choose_threshold(
        cal_val[
            target_col
        ].astype(int),
        cal_prob,
    )

    (
        X_train,
        cat_indices,
    ) = prepare_X(
        outer_train_df,
        MODEL_A_NUMERIC,
    )

    (
        X_val,
        _,
    ) = prepare_X(
        outer_val_df,
        MODEL_A_NUMERIC,
    )

    model = (
        CatBoostClassifier(
            iterations=(
                selected_iterations
            ),
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **A_FIXED_PARAMS,
        )
    )

    model.fit(
        X_train,
        outer_train_df[
            target_col
        ].astype(int),
        cat_features=(
            cat_indices
        ),
        verbose=False,
    )

    prob = (
        model.predict_proba(
            X_val
        )[:, 1]
    )

    return {
        "prob": prob,
        "threshold": threshold,
        "selected_iterations": (
            selected_iterations
        ),
        "threshold_table": (
            threshold_table
        ),
    }

## 6. Model A Original vs Audited 실행

In [33]:
model_a_data = (
    dev.copy()
)

A_VARIANTS = {
    "A_Original": (
        "matched_next_original"
    ),
    "A_Audited": (
        "matched_next_audited"
    ),
}

a_rows = []
a_pred_frames = []

for (
    variant_name,
    target_col,
) in (
    A_VARIANTS.items()
):
    for fold, val_season in enumerate(
        OUTER_VAL_SEASONS,
        start=1,
    ):
        (
            train_df,
            val_df,
        ) = split_outer(
            model_a_data,
            val_season,
        )

        result = fit_model_a(
            train_df,
            val_df,
            target_col=target_col,
            seed=(
                SEED
                + fold * 100
            ),
        )

        y = (
            val_df[
                target_col
            ]
            .astype(int)
            .to_numpy()
        )

        p_metrics = (
            class_probability_metrics(
                y,
                result[
                    "prob"
                ],
            )
        )

        t_metrics = (
            class_threshold_metrics(
                y,
                result[
                    "prob"
                ],
                result[
                    "threshold"
                ],
            )
        )

        a_rows.append({
            "variant": variant_name,
            "target_col": target_col,
            "fold": fold,
            "outer_val_season": (
                val_season
            ),
            "train_n": len(
                train_df
            ),
            "val_n": len(
                val_df
            ),
            "val_presence_rate": (
                y.mean()
            ),
            "selected_iterations": (
                result[
                    "selected_iterations"
                ]
            ),
            "threshold": (
                result[
                    "threshold"
                ]
            ),
            **p_metrics,
            **t_metrics,
        })

        pred_frame = (
            val_df[
                [
                    "row_id",
                    "player",
                    "season",
                    "target_season",
                    "team",
                    "matched_next_original",
                    "matched_next_audited",
                    "label_audit_changed",
                ]
            ]
            .copy()
        )

        pred_frame[
            "variant"
        ] = variant_name

        pred_frame[
            "presence_probability"
        ] = result[
            "prob"
        ]

        pred_frame[
            "threshold"
        ] = result[
            "threshold"
        ]

        pred_frame[
            "presence_prediction"
        ] = (
            result[
                "prob"
            ]
            >= result[
                "threshold"
            ]
        ).astype(int)

        a_pred_frames.append(
            pred_frame
        )

        print(
            f"{variant_name:<12} | "
            f"Fold {fold} {val_season} | "
            f"LogLoss {p_metrics['log_loss']:.4f} | "
            f"AUC {p_metrics['roc_auc']:.4f} | "
            f"MacroF1 {t_metrics['macro_f1']:.4f}"
        )


model_a_results = pd.DataFrame(
    a_rows
)

model_a_predictions = pd.concat(
    a_pred_frames,
    ignore_index=True,
)

A_Original   | Fold 1 2020-2021 | LogLoss 0.1988 | AUC 0.9412 | MacroF1 0.8443
A_Original   | Fold 2 2021-2022 | LogLoss 0.2921 | AUC 0.8682 | MacroF1 0.7929
A_Original   | Fold 3 2022-2023 | LogLoss 0.3338 | AUC 0.8826 | MacroF1 0.7932
A_Original   | Fold 4 2023-2024 | LogLoss 0.1914 | AUC 0.9569 | MacroF1 0.8496
A_Audited    | Fold 1 2020-2021 | LogLoss 0.1988 | AUC 0.9412 | MacroF1 0.8443
A_Audited    | Fold 2 2021-2022 | LogLoss 0.2440 | AUC 0.9078 | MacroF1 0.8099
A_Audited    | Fold 3 2022-2023 | LogLoss 0.3137 | AUC 0.8889 | MacroF1 0.8060
A_Audited    | Fold 4 2023-2024 | LogLoss 0.1879 | AUC 0.9562 | MacroF1 0.8721


In [34]:
model_a_summary = (
    model_a_results
    .groupby(
        "variant"
    )
    .agg(
        folds=(
            "fold",
            "nunique",
        ),
        log_loss_mean=(
            "log_loss",
            "mean",
        ),
        brier_mean=(
            "brier",
            "mean",
        ),
        roc_auc_mean=(
            "roc_auc",
            "mean",
        ),
        pr_auc_exit_mean=(
            "pr_auc_exit",
            "mean",
        ),
        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean",
        ),
        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
        exit_recall_mean=(
            "exit_recall",
            "mean",
        ),
        presence_recall_mean=(
            "presence_recall",
            "mean",
        ),
        threshold_mean=(
            "threshold",
            "mean",
        ),
    )
    .reset_index()
)

model_a_summary

,variant,folds,log_loss_mean,brier_mean,roc_auc_mean,pr_auc_exit_mean,balanced_accuracy_mean,macro_f1_mean,exit_recall_mean,presence_recall_mean,threshold_mean
0,A_Audited,4,0.236096,0.067393,0.923522,0.776512,0.830652,0.833073,0.709783,0.951521,0.6525
1,A_Original,4,0.254022,0.072014,0.912209,0.767071,0.821003,0.819988,0.697252,0.944755,0.6650


# Part H. Model B revalidation

## 7. Regression protocol

08-03 Fixed CatBoost S3와 동일합니다.

```text
depth = 6
learning_rate = 0.03
loss = RMSE
max iterations = 2000
early stopping = 50
eval metric = MAE
```

Outer Train 마지막 시즌에서 best iteration을 정하고
Outer Train 전체로 다시 학습합니다.

In [35]:
B_MAX_ITER = 2000
B_PATIENCE = 50

B_FIXED_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
}


def fit_model_b(
    outer_train_df,
    outer_val_df,
    target_col,
    seed,
):
    (
        cal_train,
        cal_val,
    ) = split_calibration(
        outer_train_df
    )

    (
        X_cal_train,
        cat_indices,
    ) = prepare_X(
        cal_train,
        S3_NUMERIC,
    )

    (
        X_cal_val,
        _,
    ) = prepare_X(
        cal_val,
        S3_NUMERIC,
    )

    selector = (
        CatBoostRegressor(
            iterations=B_MAX_ITER,
            eval_metric="MAE",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **B_FIXED_PARAMS,
        )
    )

    selector.fit(
        X_cal_train,
        cal_train[
            target_col
        ].astype(float),
        cat_features=(
            cat_indices
        ),
        eval_set=(
            X_cal_val,
            cal_val[
                target_col
            ].astype(float),
        ),
        early_stopping_rounds=(
            B_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    selected_iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    (
        X_train,
        cat_indices,
    ) = prepare_X(
        outer_train_df,
        S3_NUMERIC,
    )

    (
        X_val,
        _,
    ) = prepare_X(
        outer_val_df,
        S3_NUMERIC,
    )

    model = (
        CatBoostRegressor(
            iterations=(
                selected_iterations
            ),
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **B_FIXED_PARAMS,
        )
    )

    model.fit(
        X_train,
        outer_train_df[
            target_col
        ].astype(float),
        cat_features=(
            cat_indices
        ),
        verbose=False,
    )

    pred = np.clip(
        model.predict(
            X_val
        ),
        0,
        None,
    )

    return {
        "pred": pred,
        "selected_iterations": (
            selected_iterations
        ),
    }

In [36]:
def regression_metrics(
    y_true,
    pred,
):
    y = np.asarray(
        y_true,
        dtype=float,
    )

    p = np.asarray(
        pred,
        dtype=float,
    )

    out = {
        "mae": mean_absolute_error(
            y,
            p,
        ),
        "rmse": (
            mean_squared_error(
                y,
                p,
            )
            ** 0.5
        ),
        "r2": r2_score(
            y,
            p,
        ),
        "bias": float(
            np.mean(
                p - y
            )
        ),
    }

    for threshold in [
        10,
        15,
        20,
    ]:
        mask = (
            y
            >= threshold
        )

        out[
            f"{threshold}plus_n"
        ] = int(
            mask.sum()
        )

        out[
            f"{threshold}plus_mae"
        ] = (
            mean_absolute_error(
                y[
                    mask
                ],
                p[
                    mask
                ],
            )
            if mask.any()
            else np.nan
        )

        out[
            f"{threshold}plus_bias"
        ] = (
            float(
                np.mean(
                    p[
                        mask
                    ]
                    - y[
                        mask
                    ]
                )
            )
            if mask.any()
            else np.nan
        )

    return out

## 8. Model B Original vs Audited 실행

In [37]:
B_VARIANTS = {
    "B_Original": {
        "presence_col": (
            "matched_next_original"
        ),
        "target_col": (
            "next_goals_original"
        ),
    },

    "B_Audited": {
        "presence_col": (
            "matched_next_audited"
        ),
        "target_col": (
            "next_goals_audited"
        ),
    },
}

b_rows = []
b_pred_frames = []

for (
    variant_name,
    config,
) in (
    B_VARIANTS.items()
):
    population = (
        dev[
            dev[
                config[
                    "presence_col"
                ]
            ].eq(True)
        ]
        .copy()
    )

    for fold, val_season in enumerate(
        OUTER_VAL_SEASONS,
        start=1,
    ):
        (
            train_df,
            val_df,
        ) = split_outer(
            population,
            val_season,
        )

        result = fit_model_b(
            train_df,
            val_df,
            target_col=(
                config[
                    "target_col"
                ]
            ),
            seed=(
                SEED
                + fold * 1000
            ),
        )

        y = (
            val_df[
                config[
                    "target_col"
                ]
            ]
            .astype(float)
            .to_numpy()
        )

        metrics = (
            regression_metrics(
                y,
                result[
                    "pred"
                ],
            )
        )

        b_rows.append({
            "variant": variant_name,
            "presence_col": (
                config[
                    "presence_col"
                ]
            ),
            "target_col": (
                config[
                    "target_col"
                ]
            ),
            "fold": fold,
            "outer_val_season": (
                val_season
            ),
            "train_n": len(
                train_df
            ),
            "val_n": len(
                val_df
            ),
            "selected_iterations": (
                result[
                    "selected_iterations"
                ]
            ),
            **metrics,
        })

        pred_frame = (
            val_df[
                [
                    "row_id",
                    "player",
                    "season",
                    "target_season",
                    "team",
                    "matched_next_original",
                    "matched_next_audited",
                    "next_goals_original",
                    "next_goals_audited",
                    "label_audit_changed",
                    "label_audit_rule",
                ]
            ]
            .copy()
        )

        pred_frame[
            "variant"
        ] = variant_name

        pred_frame[
            "goal_prediction"
        ] = result[
            "pred"
        ]

        b_pred_frames.append(
            pred_frame
        )

        print(
            f"{variant_name:<12} | "
            f"Fold {fold} {val_season} | "
            f"N {len(val_df):,} | "
            f"MAE {metrics['mae']:.4f} | "
            f"20+ {metrics['20plus_mae']:.4f}"
        )


model_b_results = pd.DataFrame(
    b_rows
)

model_b_predictions = pd.concat(
    b_pred_frames,
    ignore_index=True,
)

Default metric period is 5 because MAE is/are not implemented for GPU


B_Original   | Fold 1 2020-2021 | N 834 | MAE 2.3362 | 20+ 9.1013


Default metric period is 5 because MAE is/are not implemented for GPU


B_Original   | Fold 2 2021-2022 | N 778 | MAE 2.4645 | 20+ 12.0112


Default metric period is 5 because MAE is/are not implemented for GPU


...

B_Audited    | Fold 2 2021-2022 | N 789 | MAE 2.4833 | 20+ 12.0112


Default metric period is 5 because MAE is/are not implemented for GPU


B_Audited    | Fold 3 2022-2023 | N 747 | MAE 2.3148 | 20+ 11.4683


Default metric period is 5 because MAE is/are not implemented for GPU


B_Audited    | Fold 4 2023-2024 | N 769 | MAE 2.4552 | 20+ 10.8335


In [38]:
model_b_summary = (
    model_b_results
    .groupby(
        "variant"
    )
    .agg(
        folds=(
            "fold",
            "nunique",
        ),
        mae_mean=(
            "mae",
            "mean",
        ),
        mae_std=(
            "mae",
            "std",
        ),
        mae_worst=(
            "mae",
            "max",
        ),
        rmse_mean=(
            "rmse",
            "mean",
        ),
        r2_mean=(
            "r2",
            "mean",
        ),
        bias_mean=(
            "bias",
            "mean",
        ),
        tenplus_mae_mean=(
            "10plus_mae",
            "mean",
        ),
        fifteenplus_mae_mean=(
            "15plus_mae",
            "mean",
        ),
        twentyplus_mae_mean=(
            "20plus_mae",
            "mean",
        ),
    )
    .reset_index()
)

model_b_summary

,variant,folds,mae_mean,mae_std,mae_worst,rmse_mean,r2_mean,bias_mean,tenplus_mae_mean,fifteenplus_mae_mean,twentyplus_mae_mean
0,B_Audited,4,2.397380,0.084225,2.483263,3.444320,0.457201,-0.014946,6.431992,8.627918,10.853575
1,B_Original,4,2.387612,0.078865,2.464486,3.435972,0.458169,-0.007529,6.392789,8.635947,10.890876


# Part I. 직접 영향 row

## 9. 보정된 20명이 실제 평가 Fold에 몇 명 들어오는가?

보정 row가 어느 Fold의 Model B validation population에 새로 들어왔는지 봅니다.

In [39]:
changed_rows = (
    dev[
        dev[
            "label_audit_changed"
        ].eq(1)
    ][
        [
            "row_id",
            "player",
            "season",
            "target_season",
            "team",
            "next_goals_original",
            "next_goals_audited",
            "label_audit_rule",
        ]
    ]
    .copy()
)

changed_rows[
    "is_outer_validation"
] = (
    changed_rows[
        "season"
    ].isin(
        OUTER_VAL_SEASONS
    )
)

changed_rows

,row_id,player,season,target_season,team,next_goals_original,next_goals_audited,label_audit_rule,is_outer_validation
1063,20600,Amine Adli,2021-2022,2022-2023,Leverkusen,0.0,5.0,exact_name_age,True
1066,20601,Amine Gouiri,2021-2022,2022-2023,Nice,0.0,15.0,exact_name_age,True
2098,21587,Atakan Karazor,2022-2023,2023-2024,Stuttgart,0.0,0.0,exact_name_age,True
2675,21611,Brahim Díaz,2022-2023,2023-2024,Milan,0.0,8.0,exact_name_age,True
4981,20735,Demarai Gray,2021-2022,2022-2023,Everton,0.0,4.0,exact_name_age,True
5824,21733,Elvis Rexhbeçaj,2022-2023,2023-2024,Augsburg,0.0,2.0,exact_name_age,True
6165,21752,Evann Guessand,2022-2023,2023-2024,Nantes,0.0,6.0,exact_name_age,True
7401,21795,Gabriel Strefezza,2022-2023,2023-2024,Lecce,0.0,1.0,exact_name_age,True
8726,20885,Hugo Guillamón,2021-2022,2022-2023,Valencia,0.0,1.0,exact_name_age,True
8727,21844,Hugo Guillamón,2022-2023,2023-2024,Valencia,0.0,1.0,exact_name_age,True


## 10. Audited Model B가 새로 평가한 correction rows

In [40]:
audited_changed_predictions = (
    model_b_predictions[
        model_b_predictions[
            "variant"
        ].eq(
            "B_Audited"
        )
        & model_b_predictions[
            "label_audit_changed"
        ].eq(1)
    ]
    .copy()
)

if not audited_changed_predictions.empty:
    audited_changed_predictions[
        "abs_error"
    ] = (
        audited_changed_predictions[
            "goal_prediction"
        ]
        - audited_changed_predictions[
            "next_goals_audited"
        ]
    ).abs()

    display(
        audited_changed_predictions[
            [
                "player",
                "season",
                "target_season",
                "team",
                "next_goals_audited",
                "goal_prediction",
                "abs_error",
                "label_audit_rule",
            ]
        ]
        .sort_values(
            "next_goals_audited",
            ascending=False,
        )
    )

else:
    print(
        "Outer validation에 새 correction row가 없습니다."
    )

,player,season,target_season,team,next_goals_audited,goal_prediction,abs_error,label_audit_rule
4620,Sepe Elye Wahi,2021-2022,2022-2023,Montpellier,19.0,6.597163,12.402837,same_team_alias_age
3987,Amine Gouiri,2021-2022,2022-2023,Nice,15.0,9.277918,5.722082,exact_name_age
4417,Martinelli,2021-2022,2022-2023,Arsenal,15.0,5.029779,9.970221,same_team_alias_age
4832,Brahim Díaz,2022-2023,2023-2024,Milan,8.0,5.144673,2.855327,exact_name_age
4941,Evann Guessand,2022-2023,2023-2024,Nantes,6.0,3.023973,2.976027,exact_name_age
4358,Lee Kangin,2021-2022,2022-2023,Mallorca,6.0,1.676346,4.323654,same_team_alias_age
4292,Jonathan Bamba,2021-2022,2022-2023,Lille,6.0,3.071979,2.928021,exact_name_age
3986,Amine Adli,2021-2022,2022-2023,Leverkusen,5.0,3.475937,1.524063,exact_name_age
5051,Jean-Ricner Bellegarde,2022-2023,2023-2024,Strasbourg,4.0,2.536073,1.463927,exact_name_age
4103,Demarai Gray,2021-2022,2022-2023,Everton,4.0,4.126974,0.126974,exact_name_age


# Part J. Delta summary

In [41]:
def summary_delta(
    table,
    original_name,
    audited_name,
    metrics,
):
    original = (
        table[
            table[
                "variant"
            ].eq(
                original_name
            )
        ]
        .iloc[0]
    )

    audited = (
        table[
            table[
                "variant"
            ].eq(
                audited_name
            )
        ]
        .iloc[0]
    )

    rows = []

    for metric in (
        metrics
    ):
        rows.append({
            "metric": metric,
            "original": (
                original[
                    metric
                ]
            ),
            "audited": (
                audited[
                    metric
                ]
            ),
            "delta_audited_minus_original": (
                audited[
                    metric
                ]
                - original[
                    metric
                ]
            ),
        })

    return pd.DataFrame(
        rows
    )


model_a_delta = summary_delta(
    model_a_summary,
    "A_Original",
    "A_Audited",
    [
        "log_loss_mean",
        "brier_mean",
        "roc_auc_mean",
        "pr_auc_exit_mean",
        "macro_f1_mean",
        "exit_recall_mean",
    ],
)

model_b_delta = summary_delta(
    model_b_summary,
    "B_Original",
    "B_Audited",
    [
        "mae_mean",
        "rmse_mean",
        "r2_mean",
        "bias_mean",
        "tenplus_mae_mean",
        "fifteenplus_mae_mean",
        "twentyplus_mae_mean",
    ],
)

print(
    "Model A delta"
)

display(
    model_a_delta
)

print(
    "Model B delta"
)

display(
    model_b_delta
)

Model A delta


,metric,original,audited,delta_audited_minus_original
0,log_loss_mean,0.254022,0.236096,-0.017926
1,brier_mean,0.072014,0.067393,-0.004621
2,roc_auc_mean,0.912209,0.923522,0.011314
3,pr_auc_exit_mean,0.767071,0.776512,0.009441
4,macro_f1_mean,0.819988,0.833073,0.013085
5,exit_recall_mean,0.697252,0.709783,0.012531


Model B delta


,metric,original,audited,delta_audited_minus_original
0,mae_mean,2.387612,2.397380,0.009768
1,rmse_mean,3.435972,3.444320,0.008348
2,r2_mean,0.458169,0.457201,-0.000967
3,bias_mean,-0.007529,-0.014946,-0.007417
4,tenplus_mae_mean,6.392789,6.431992,0.039204
5,fifteenplus_mae_mean,8.635947,8.627918,-0.008029
6,twentyplus_mae_mean,10.890876,10.853575,-0.037300


# Part K. 판단 기준

## Model A

Audited label 반영 후:

- LogLoss / Brier가 유지 또는 개선되는가?
- ROC-AUC가 유지되는가?
- Exit Recall / Macro F1이 무너지지 않는가?

20건 수정만으로 성능이 크게 움직이면
label noise 민감도가 높다는 뜻입니다.

## Model B

Audited population 반영 후:

- 전체 MAE가 크게 변하는가?
- 10+/15+/20+가 얼마나 변하는가?
- 새로 복원된 15~19골 사례 때문에 high-scorer 문제가 더 명확해지는가?

---

# 중요한 해석

09-01A의 보정은 모델 점수를 높이기 위한 작업이 아닙니다.

잘못 0골로 처리된 선수를 실제 득점으로 되돌리면
오히려 모델 점수가 **나빠질 수도 있습니다.**

그 경우:

> 모델이 어려운 사례를 평가에서 빠뜨리고 있었기 때문에
> 기존 점수가 다소 낙관적이었다

는 의미가 됩니다.

# Part L. 결과 저장

In [42]:
OUTPUTS = {
    "model_a_results": (
        ARTIFACT_DIR
        / "09_01B_model_a_outer_results.csv"
    ),
    "model_a_summary": (
        ARTIFACT_DIR
        / "09_01B_model_a_summary.csv"
    ),
    "model_a_delta": (
        ARTIFACT_DIR
        / "09_01B_model_a_delta.csv"
    ),
    "model_a_predictions": (
        ARTIFACT_DIR
        / "09_01B_model_a_predictions.csv"
    ),
    "model_b_results": (
        ARTIFACT_DIR
        / "09_01B_model_b_outer_results.csv"
    ),
    "model_b_summary": (
        ARTIFACT_DIR
        / "09_01B_model_b_summary.csv"
    ),
    "model_b_delta": (
        ARTIFACT_DIR
        / "09_01B_model_b_delta.csv"
    ),
    "model_b_predictions": (
        ARTIFACT_DIR
        / "09_01B_model_b_predictions.csv"
    ),
    "impact_summary": (
        ARTIFACT_DIR
        / "09_01B_label_impact_summary.csv"
    ),
    "stable_rank_diag": (
        ARTIFACT_DIR
        / "09_01B_stable_review_rank_diagnostic.csv"
    ),
}


model_a_results.to_csv(
    OUTPUTS[
        "model_a_results"
    ],
    index=False,
)

model_a_summary.to_csv(
    OUTPUTS[
        "model_a_summary"
    ],
    index=False,
)

model_a_delta.to_csv(
    OUTPUTS[
        "model_a_delta"
    ],
    index=False,
)

model_a_predictions.to_csv(
    OUTPUTS[
        "model_a_predictions"
    ],
    index=False,
)

model_b_results.to_csv(
    OUTPUTS[
        "model_b_results"
    ],
    index=False,
)

model_b_summary.to_csv(
    OUTPUTS[
        "model_b_summary"
    ],
    index=False,
)

model_b_delta.to_csv(
    OUTPUTS[
        "model_b_delta"
    ],
    index=False,
)

model_b_predictions.to_csv(
    OUTPUTS[
        "model_b_predictions"
    ],
    index=False,
)

impact_summary.to_csv(
    OUTPUTS[
        "impact_summary"
    ],
    index=False,
)

stable_rank_diag.to_csv(
    OUTPUTS[
        "stable_rank_diag"
    ],
    index=False,
)


print(
    "Saved:"
)

for name, path in (
    OUTPUTS.items()
):
    print(
        f"- {name:<24}",
        path.resolve(),
    )

Saved:
- model_a_results          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_a_outer_results.csv
- model_a_summary          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_a_summary.csv
- model_a_delta            /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_a_delta.csv
- model_a_predictions      /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_a_predictions.csv
- model_b_results          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_b_outer_results.csv
- model_b_summary          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_b_summary.csv
- model_b_delta            /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_b_delta.csv
- model_b_predictions      /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_model_b_predictions.csv
- impact_summary         

## 11. Protocol

In [43]:
PROTOCOL = {
    "stage": (
        "09-01B Audited Label Revalidation"
    ),

    "source": (
        "09_01A_snapshot_conservative_labels.csv"
    ),

    "audit_policy": (
        "only 09-01A hard corrections are applied"
    ),

    "manual_cross_team_alias_added": (
        False
    ),

    "model_a": {
        "type": (
            "Fixed CatBoost Classifier"
        ),
        "features": (
            "S3 + Model A transfer_event_preseason "
            "+ destination_in_big5"
        ),
        "original_target": (
            "matched_next_original"
        ),
        "audited_target": (
            "matched_next_audited"
        ),
    },

    "model_b": {
        "type": (
            "Fixed CatBoost Regressor S3"
        ),
        "params": (
            B_FIXED_PARAMS
        ),
        "original_population": (
            "matched_next_original == True"
        ),
        "audited_population": (
            "matched_next_audited == True"
        ),
        "original_target": (
            "next_goals_original"
        ),
        "audited_target": (
            "next_goals_audited"
        ),
    },

    "outer_validation_seasons": (
        OUTER_VAL_SEASONS
    ),

    "test_input_season": (
        LOCKED_TEST_INPUT_SEASON
    ),

    "test_target_season": (
        LOCKED_TEST_TARGET_SEASON
    ),

    "test_status": (
        "LOCKED / NOT LOADED"
    ),

    "next_step": (
        "Decide audited labels and Model A/B baseline; "
        "then build 09-02 Two-stage Integration."
    ),
}

PROTOCOL_PATH = (
    ARTIFACT_DIR
    / "09_01B_protocol.json"
)

with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PROTOCOL,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    PROTOCOL_PATH.resolve()
)

/content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01B_protocol.json


# 실행 후 보내줄 파일

우선 아래 6개만 보내면 됩니다.

```text
09_01B_label_impact_summary.csv
09_01B_model_a_summary.csv
09_01B_model_a_delta.csv
09_01B_model_b_summary.csv
09_01B_model_b_delta.csv
09_01B_model_b_predictions.csv
```

그 결과를 보고:

```text
09-01B
↓
audited label 최종 채택
↓
09-02 Two-stage Integration
```

으로 넘어갑니다.